### **PDF Document Processing (Structured & Visual)**
To effectively ingest PDFs into your vector database, follow this pipeline:

* **Text Extraction:** Use libraries like `PyMuPDF` or `Unstructured` to pull text while preserving font sizes and bold headers (this helps with semantic chunking).
* **Visual-to-Text (OCR/VLM):** * For **Images/Charts**, use a Vision Model (like GPT-4o or Claude 3.5 Sonnet) to describe the graphic.
    * *Example prompt:* "Describe this technical chart's axes, data points, and core conclusion in detail."
* **Chunking Strategy:** Instead of fixed-size chunks (e.g., 500 characters), use **Recursive Character Splitting** to ensure sentences aren't cut in half.

### **Email Q&A Processing (Unstructured Dialogue)**
Emails are often "noisy" (signatures, "Best regards", irrelevant small talk). You need a pre-processing agent:

* **Knowledge Extraction:** Feed the email thread to an LLM to distill it.
    * *System Prompt:* "Extract the core technical problem and the final verified solution from this email thread. Format as a concise FAQ pair."
* **Contextual Metadata:** Store the "Project Name" or "Date" as metadata so the retriever can prioritize the most recent information.
* **Deduplication:** Check if a similar Q&A already exists in the vector DB to avoid redundant or conflicting answers.

In [42]:
from pathlib import Path

# 当前文件所在路径（notebook / script 都适用）
BASE_DIR = Path.cwd()

# 自动找到 project root
if (BASE_DIR / "data").exists():
    project_root = BASE_DIR
else:
    project_root = BASE_DIR.parent

# 你的目标目录
md_dir = project_root / "data/processed/pdf_parsed"

print("Project root:", project_root)
print("MD dir:", md_dir)
print("Exists:", md_dir.exists())

Project root: /Users/promab/anaconda_projects/email_agent
MD dir: /Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed
Exists: True


In [44]:
md_files = list(md_dir.glob("*.md"))
print(f"Found {len(md_files)} markdown files")

Found 1 markdown files


In [52]:
from langchain_community.document_loaders import TextLoader

all_docs = []

for md_file in md_files:
    loader = TextLoader(str(md_file), encoding="utf-8")
    docs = loader.load()

    for d in docs:
        d.metadata["file_name"] = md_file.name
        d.metadata["source_path"] = str(md_file)

    all_docs.extend(docs)

print(f"Total docs loaded: {len(all_docs)}")

Total docs loaded: 1


In [53]:
doc = all_docs[0]
print(doc.page_content)
print(doc.metadata)

Baculovirus Protein Expression Services

## **Service Features**

The Baculovirus Protein Expression Service offers the following features:

* High levels of protein expression  
* Competitive pricing  
* DNA design included  
* Fast turnaround time

## **Breakdown of Standard Service by Phase**

The standard service of Baculovirus Protein Expression is divided into three phases.

### **Phase I**

This phase includes:

* Amplification/isolation of the gene of interest (GOI) from a customer-supplied construct  
* Subcloning into a transfer vector with His-tag or GST-tag  
* Sequence confirmation of insert

Estimated time: 2–3 weeks  
Cost: $1,500  
Payment timing: due at the beginning of the phase

### **Phase II**

This phase includes:

* Transformation of baculoviral DNA into *E. coli* with the gene of interest to produce recombinant bacmid/GOI DNA  
* Culture of transformed *E. coli* and extraction of recombinant bacmid (recombinant viral DNA)  
* Transformation of Sf9 insect cells w

In [54]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("##", "section"),
    ("###", "subsection"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

split_docs = []

for doc in all_docs:
    pieces = markdown_splitter.split_text(doc.page_content)

    for p in pieces:
        p.metadata["file_name"] = doc.metadata.get("file_name", "")
        p.metadata["source_path"] = doc.metadata.get("source_path", "")

    split_docs.extend(pieces)

print(f"Total split docs: {len(split_docs)}")

Total split docs: 16


In [55]:
for i, d in enumerate(split_docs):
    print(f"\n--- Chunk {i} ---")
    print("Metadata:", d.metadata)
    print(d.page_content[:500])


--- Chunk 0 ---
Metadata: {'file_name': 'Baculovirus Protein Expression.md', 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/Baculovirus Protein Expression.md'}
Baculovirus Protein Expression Services

--- Chunk 1 ---
Metadata: {'section': '**Service Features**', 'file_name': 'Baculovirus Protein Expression.md', 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/Baculovirus Protein Expression.md'}
The Baculovirus Protein Expression Service offers the following features:  
* High levels of protein expression
* Competitive pricing
* DNA design included
* Fast turnaround time

--- Chunk 2 ---
Metadata: {'section': '**Breakdown of Standard Service by Phase**', 'file_name': 'Baculovirus Protein Expression.md', 'source_path': '/Users/promab/anaconda_projects/email_agent/data/processed/pdf_parsed/Baculovirus Protein Expression.md'}
The standard service of Baculovirus Protein Expression is divided into three phases.

---